# Session 07: FastAPI

**Notebook 1 of 2: Concepts** · used during the live session and for revision afterwards (about 60 minutes on your own)

In Session 05 you **called** an HTTP API; in Session 06 you described data with Pydantic. This session puts the two together from the other side: you'll **serve** an HTTP API with **FastAPI**, using Pydantic models as the contract for every request and response.

By the end of this notebook you will be able to:

1. **Implement** endpoints with path and query parameters, request bodies and response models.
2. **Return** the right status codes, with `HTTPException` and with exception handlers for your own exceptions.
3. **Inject** settings and services with `Depends`, and **override** them in tests.
4. **Structure** an application into routers, schemas and a service layer, built by an app factory.
5. **Test** endpoints with `TestClient`, and **read** the OpenAPI document FastAPI generates.

> **Kernel:** the course environment, after `uv sync`. FastAPI is in its `web` group. Everything here runs offline: requests go to the app in memory, through `TestClient`, without opening a network port.

## The engineering problem

The Orbital Logistics (fictional) incident report from Project 01 is a JSON file written at 02:00 every night. Three teams now want more:

- The **service desk** wants to look up a ticket *now*, not as of last night.
- The **monitoring team** wants to **create** tickets automatically when an alert fires.
- The **on-call dashboard** wants open P1 tickets for one team, a page at a time.

Sharing a file can't do that. They need an HTTP API, with documentation they can read without asking you, and errors their code can act on: a `404` for an unknown ticket, a `409` for a duplicate, and a `422` that says exactly which field was wrong.

In Java you'd reach for Spring Boot, in C# for ASP.NET Core, in Node.js for Express or NestJS. In Python, for a JSON API in 2026, the usual answer is **FastAPI**.

## Part 1: A first endpoint

A FastAPI application is an object, and an endpoint is an ordinary function registered on it with a decorator that names the HTTP method and path:

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI(title="Asset Inventory")


@app.get("/health")
def health() -> dict[str, str]:
    return {"status": "ok"}


client = TestClient(app)  # an HTTP client wired directly to the app: no server, no port
response = client.get("/health")
print(response.status_code, response.headers["content-type"], response.json())

**Explain:**

- `@app.get("/health")` registers `health` for `GET /health`. There are matching decorators for `post`, `put`, `patch` and `delete`.
- The return value is converted to JSON automatically. You don't call a serialiser or set a content type.
- **`TestClient`** is an HTTPX-style client (the same `get`, `post`, `json()` and `status_code` you used in Session 05) that sends requests straight into the application object. Tests run in milliseconds and need no free port.

| | Spring Boot | ASP.NET Core | Express | FastAPI |
|---|---|---|---|---|
| Register an endpoint | `@GetMapping("/health")` | `app.MapGet("/health", ...)` or `[HttpGet]` | `app.get("/health", handler)` | `@app.get("/health")` |
| Path parameter | `@PathVariable String id` | `{id}` bound to a parameter | `req.params.id` (a string) | a typed function parameter |
| Request body | `@RequestBody @Valid Dto dto` | `[FromBody] Dto dto` | `req.body`, then validate it yourself | a parameter typed as a Pydantic model |
| Dependency injection | the Spring container | the built-in container | none built in | `Depends(...)`, resolved per request |
| API documentation | springdoc add-on | Swagger add-on | add-on | built in, at `/docs` |
| Server | embedded Tomcat | Kestrel | Node's `http` | **Uvicorn**, a separate ASGI server |

The last row matters: FastAPI is a framework, not a server. In production, **Uvicorn** accepts connections and passes each request to the app through **ASGI**, a standard interface between Python web servers and frameworks. You'll start it with `fastapi dev` in the workbook.

![Life of a request in FastAPI](../../../assets/diagrams/fastapi-request-flow.svg)

**How this works**

- Uvicorn parses the HTTP request and hands it to the app. FastAPI first **routes** it: no matching path gives `404`, and a matching path with the wrong method gives `405`.
- FastAPI then resolves the endpoint's **dependencies** and **validates** its inputs (path, query string, headers and body) with Pydantic, collecting every problem. Any problem means `422`, and your function is **never called**, so its code can trust its arguments.
- Your function runs. If it raises an exception that has a registered **handler**, the handler builds the response (`404`, `409` ...). Any other exception becomes a `500`.
- The return value is validated against the **response model** and serialised to JSON, with the status code the route declares.
- Every arrow on the right is a response the client can act on. Designing an API largely means deciding which failures get which status, and what the body says.

## Part 2: Path and query parameters

Parameters are declared as **typed function parameters**. FastAPI decides where each comes from:

- a name that appears in the path (`{asset_id}`) is a **path parameter**;
- a simple type (`str`, `int`, `bool` ...) that isn't in the path is a **query parameter**;
- a Pydantic model is the **request body** (Part 3).

In [ ]:
from typing import Annotated, Literal

from fastapi import Path, Query

ASSETS = {
    "db-03": {"id": "db-03", "team": "database", "criticality": "high"},
    "vpn-ber": {"id": "vpn-ber", "team": "network", "criticality": "medium"},
    "wiki": {"id": "wiki", "team": "it-support", "criticality": "low"},
}

app = FastAPI(title="Asset Inventory")


@app.get("/assets/{asset_id}")
def get_asset(asset_id: Annotated[str, Path(pattern=r"^[a-z0-9-]+$", max_length=30)]) -> dict:
    return ASSETS[asset_id]


@app.get("/assets")
def list_assets(
    team: str | None = None,
    criticality: Literal["low", "medium", "high"] | None = None,
    limit: Annotated[int, Query(ge=1, le=100)] = 20,
) -> list[dict]:
    found = [
        asset
        for asset in ASSETS.values()
        if (team is None or asset["team"] == team)
        and (criticality is None or asset["criticality"] == criticality)
    ]
    return found[:limit]


client = TestClient(app)
print(client.get("/assets/db-03").json())
print(client.get("/assets", params={"team": "network"}).json())
print(client.get("/assets?criticality=high&limit=1").json())

`Annotated[int, Query(ge=1, le=100)]` attaches **constraints** to a parameter, using the same rules as Pydantic's `Field` in Session 06. `Path(...)` does the same for path parameters. A parameter with a default is optional; without one it's required.

**Predict:** what does each of these requests return?

1. `GET /assets?limit=abc`
2. `GET /assets?limit=0&criticality=urgent`
3. `GET /assets/DB_03`

In [ ]:
for url in ["/assets?limit=abc", "/assets?limit=0&criticality=urgent", "/assets/DB_03"]:
    response = client.get(url)
    print(url, "->", response.status_code)
    for error in response.json()["detail"]:
        print(f"    {error['loc']}  {error['type']}: {error['msg']}")

**Explain:** all three are `422 Unprocessable Content`, and the body lists **every** problem with its location: `["query", "limit"]`, `["path", "asset_id"]`. That's Pydantic's `errors()` from Session 06, delivered over HTTP. Notice that `get_asset` was never called with `DB_03`; validation stopped it first.

Query strings are text, so FastAPI converts them in **lax** mode: `"5"` becomes `5`, and `"true"`, `"1"`, `"yes"` and `"on"` all become `True` for a `bool`.

**Predict:** what does `GET /assets/printer-7` return? It matches the pattern, but there's no such asset.

In [ ]:
try:
    client.get("/assets/printer-7")
except KeyError as error:
    print("TestClient re-raised the server's exception:", repr(error))

lenient_client = TestClient(app, raise_server_exceptions=False)  # behave like a real client
response = lenient_client.get("/assets/printer-7")
print(response.status_code, response.text)

**Explain:** `ASSETS["printer-7"]` raised `KeyError`, which nothing handled, so the server answered **`500 Internal Server Error`**, a response that tells the client "the server is broken" when actually the client asked for something that doesn't exist. By default, `TestClient` re-raises server exceptions so that your tests show the real traceback; `raise_server_exceptions=False` shows what an HTTP client would see. Part 4 fixes this endpoint.

### Many query parameters: a model

When an endpoint has several filters, declare them as a Pydantic model and mark it `Query()`. With `extra="forbid"`, a misspelt parameter becomes a `422` instead of being silently ignored:

In [ ]:
from pydantic import BaseModel, ConfigDict, Field


class AssetFilters(BaseModel):
    model_config = ConfigDict(extra="forbid")

    team: str | None = None
    criticality: Literal["low", "medium", "high"] | None = None
    limit: int = Field(default=20, ge=1, le=100)


@app.get("/v2/assets")
def list_assets_v2(filters: Annotated[AssetFilters, Query()]) -> list[dict]:
    found = [a for a in ASSETS.values() if filters.team in (None, a["team"])]
    return found[: filters.limit]


print(client.get("/v2/assets", params={"team": "database"}).json())
response = client.get("/v2/assets", params={"teem": "database"})
print(
    response.status_code, response.json()["detail"][0]["msg"], response.json()["detail"][0]["loc"]
)

**Engineering Insight:** without `extra="forbid"`, `?teem=database` returns **every** asset, unfiltered, with `200 OK`. The caller's typo turns into a wrong answer that looks right. For filters, failing loudly is kinder. The workbook's `TicketFilters` uses this pattern.

## Part 3: Request bodies and response models

### Request bodies

A parameter typed as a Pydantic model is read from the **JSON body**. Validation is the model's own, so everything from Session 06 applies: types, constraints, `Literal`, strict fields and validators.

In [ ]:
from fastapi import status


class NewAsset(BaseModel):
    id: str = Field(pattern=r"^[a-z0-9-]+$")
    name: str = Field(min_length=1)
    team: str
    criticality: Literal["low", "medium", "high"] = "medium"


app = FastAPI(title="Asset Inventory")
inventory: dict[str, NewAsset] = {}


@app.post("/assets", status_code=status.HTTP_201_CREATED)
def create_asset(asset: NewAsset) -> NewAsset:
    inventory[asset.id] = asset
    return asset


client = TestClient(app)
response = client.post("/assets", json={"id": "db-04", "name": "Orders DB", "team": "database"})
print(response.status_code, response.json())

response = client.post("/assets", json={"id": "DB 04", "team": "database", "criticality": "urgent"})
print(response.status_code, [(e["loc"], e["type"]) for e in response.json()["detail"]])

**Explain:** a successful `POST` that creates something returns **`201 Created`**, not `200`; `status_code=` on the decorator says so. The `status` module names every code (`status.HTTP_201_CREATED`), which reads better than a bare number. Errors in the body are located as `["body", "<field>"]`.

### Response models: the public contract

The **return type** of an endpoint is its **response model**. FastAPI validates the return value against it and **serialises only the fields the model declares**. That makes the response model a filter, as well as documentation:

In [ ]:
class AssetRecord(BaseModel):
    """What the service stores internally."""

    id: str
    name: str
    team: str
    on_call_phone: str  # internal only
    admin_password_hint: str  # must never leave the service


class AssetOut(BaseModel):
    """What the API promises to return."""

    id: str
    name: str
    team: str


RECORDS = {
    "db-03": AssetRecord(
        id="db-03",
        name="Orders DB replica",
        team="database",
        on_call_phone="+00 0000 000000",
        admin_password_hint="(synthetic)",
    ),
}

app = FastAPI(title="Asset Inventory")


@app.get("/leaky/{asset_id}")
def leaky(asset_id: str) -> AssetRecord:
    return RECORDS[asset_id]


@app.get("/assets/{asset_id}")
def get_asset(asset_id: str) -> AssetOut:
    return RECORDS[asset_id]  # an AssetRecord, converted to AssetOut on the way out


client = TestClient(app)
print("leaky:", client.get("/leaky/db-03").json())
print("safe: ", client.get("/assets/db-03").json())

**Explain:** returning the internal record directly published every field, including the ones that should never leave the service. With `AssetOut` as the return type, the same object is filtered down to the public fields. This is why the workbook has **separate API schemas** (`TicketOut`) next to the domain models: the domain can gain internal fields without changing, or leaking through, the API.

**Modify:** add `criticality: str = "unknown"` to `AssetOut`. What appears in the response, and why?

Two equivalent ways to declare the response model:

| Style | When |
|---|---|
| Return type: `def get_asset(...) -> AssetOut` | The default: one declaration serves the type checker and FastAPI |
| `@app.get(..., response_model=AssetOut)` | When the function returns something else on purpose, such as a dict or a `Response` object |

## Part 4: Status codes and errors

Every response is a message to another program. The status code says *what kind* of outcome it is; the body says *which*.

| Code | Meaning | Typical cause |
|---|---|---|
| `200 OK` | Success, with a body | `GET`, or an update |
| `201 Created` | A new resource was created | `POST` |
| `204 No Content` | Success, no body | `DELETE` |
| `400 Bad Request` | The request is wrong in a way validation can't express | Rarely needed with FastAPI |
| `401` / `403` | Not authenticated / not allowed | A missing or wrong token; the wrong role |
| `404 Not Found` | No such resource | An unknown id |
| `409 Conflict` | The request conflicts with the current state | A duplicate id |
| `422 Unprocessable Content` | The request is well-formed but invalid | FastAPI's validation |
| `500 Internal Server Error` | The server failed | An unhandled exception: a bug |

### `HTTPException`: raise a status from an endpoint

In [ ]:
from fastapi import HTTPException

app = FastAPI(title="Asset Inventory")


@app.get("/assets/{asset_id}")
def get_asset(asset_id: str) -> AssetOut:
    if asset_id not in RECORDS:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND, detail=f"asset {asset_id} not found"
        )
    return RECORDS[asset_id]


client = TestClient(app)
response = client.get("/assets/printer-7")
print(response.status_code, response.json())

`HTTPException` is the quickest fix, and fine inside a route. But it's an **HTTP** exception, so it belongs only in the API layer. Code that also runs from a command line or a background job shouldn't know about status codes.

### Exception handlers: map your own exceptions to responses

The service layer raises **domain** exceptions; the API layer decides what they mean in HTTP, once, in a handler:

In [ ]:
from fastapi import Request
from fastapi.responses import JSONResponse


class AssetNotFoundError(Exception):
    def __init__(self, asset_id: str) -> None:
        super().__init__(f"asset {asset_id} not found")
        self.asset_id = asset_id


def find_asset(asset_id: str) -> AssetRecord:
    """Service code: no HTTP in sight."""
    try:
        return RECORDS[asset_id]
    except KeyError:
        raise AssetNotFoundError(asset_id) from None


app = FastAPI(title="Asset Inventory")


@app.get("/assets/{asset_id}")
def get_asset(asset_id: str) -> AssetOut:
    return find_asset(asset_id)


def asset_not_found(request: Request, error: AssetNotFoundError) -> JSONResponse:
    return JSONResponse(status_code=status.HTTP_404_NOT_FOUND, content={"detail": str(error)})


app.add_exception_handler(AssetNotFoundError, asset_not_found)

client = TestClient(app)
print(client.get("/assets/printer-7").status_code, client.get("/assets/printer-7").json())

**Explain:** the route is one line and has no error handling at all. Every endpoint that calls `find_asset` now returns the same `404` with the same body shape, and a new endpoint can't forget. The `{"detail": ...}` shape matches what `HTTPException` and FastAPI's own errors use, so clients see one convention.

This is the same idea as Spring's `@ExceptionHandler` / `@ControllerAdvice`, or ASP.NET Core's exception-handling middleware.

**Common Mistake:** a broad handler for `Exception` that returns `200` or hides the error. Unexpected exceptions *should* become `500`, and should be logged with their traceback. Handle the exceptions you expect, and let the rest fail loudly.

## Part 5: Dependency injection with `Depends`

Endpoints need things that aren't part of the request: settings, a service, a database session, the current user. FastAPI's answer is **dependencies**: functions that FastAPI calls for each request, whose return value is passed to the endpoint.

In [ ]:
from functools import lru_cache

from fastapi import Depends
from pydantic_settings import BaseSettings, SettingsConfigDict


class InventorySettings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="INVENTORY_", extra="ignore")

    default_page_size: int = 20
    read_only: bool = False


@lru_cache
def get_settings() -> InventorySettings:
    return InventorySettings()  # built once, on first use, then reused


class AssetService:
    def __init__(self, records: dict[str, AssetRecord]) -> None:
        self._records = records

    def get(self, asset_id: str) -> AssetRecord:
        try:
            return self._records[asset_id]
        except KeyError:
            raise AssetNotFoundError(asset_id) from None

    def count(self) -> int:
        return len(self._records)


SERVICE = AssetService(RECORDS)


def get_service() -> AssetService:
    return SERVICE


SettingsDep = Annotated[InventorySettings, Depends(get_settings)]
ServiceDep = Annotated[AssetService, Depends(get_service)]

app = FastAPI(title="Asset Inventory")
app.add_exception_handler(AssetNotFoundError, asset_not_found)


@app.get("/assets/{asset_id}")
def get_asset(asset_id: str, service: ServiceDep) -> AssetOut:
    return service.get(asset_id)


@app.get("/status")
def service_status(service: ServiceDep, settings: SettingsDep) -> dict:
    return {"assets": service.count(), "read_only": settings.read_only}


client = TestClient(app)
print(client.get("/assets/db-03").json())
print(client.get("/status").json())

**Explain:**

- `Annotated[AssetService, Depends(get_service)]` says "this parameter is an `AssetService`, and FastAPI gets it by calling `get_service`". Naming the alias (`ServiceDep`) keeps signatures short and the wiring in one place.
- The endpoint never constructs its collaborators. It **declares** what it needs, like constructor injection in Spring or ASP.NET Core, but resolved **per request, by calling functions**, with no container or registration step.
- `@lru_cache` on `get_settings` makes it build `InventorySettings` once. Without it, every request would re-read the environment.
- A dependency can itself have parameters, including other dependencies and request data such as headers.

### A dependency that guards endpoints

Dependencies can also **reject** a request before the endpoint runs. Here's a minimal API-key check (Session 05's client sent exactly this kind of header):

In [ ]:
from fastapi import Header

API_KEYS = {"not-a-real-key"}  # in a real service: from settings, as a SecretStr


def require_api_key(x_api_key: Annotated[str | None, Header()] = None) -> str:
    if x_api_key not in API_KEYS:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED, detail="missing or invalid API key"
        )
    return x_api_key


@app.get("/admin/assets", dependencies=[Depends(require_api_key)])
def admin_list(service: ServiceDep) -> dict:
    return {"assets": service.count()}


print(client.get("/admin/assets").status_code)
print(client.get("/admin/assets", headers={"X-API-Key": "not-a-real-key"}).json())

The parameter `x_api_key` is read from the `X-API-Key` header: FastAPI converts underscores to hyphens and ignores case. `dependencies=[...]` runs a dependency for its side effect (here, possibly rejecting the request) when the endpoint doesn't need its return value.

Real authentication (OAuth2, JWTs, user accounts) is beyond this course. The pattern is the same: a dependency that returns the authenticated user or raises `401`.

### Overriding dependencies in tests

Because endpoints receive their collaborators instead of building them, a test can **substitute** them. `app.dependency_overrides` maps the original dependency function to a replacement:

In [ ]:
fake_service = AssetService(
    {
        "test-1": AssetRecord(
            id="test-1", name="Test asset", team="qa", on_call_phone="-", admin_password_hint="-"
        ),
    }
)

app.dependency_overrides[get_service] = lambda: fake_service
app.dependency_overrides[get_settings] = lambda: InventorySettings(read_only=True)
try:
    print(client.get("/assets/test-1").json())
    print(client.get("/status").json())
    print(client.get("/assets/db-03").status_code, "(the real data isn't visible)")
finally:
    app.dependency_overrides.clear()  # never let an override leak into the next test

print(client.get("/status").json())

**Engineering Insight:** this is why the workbook's routes take the service through `Depends(get_service)` and never create it. In a test you can swap in a service with three hand-made tickets, a service that raises, or (in Session 08) one whose HTTP client is mocked, without touching the routes. Always clear overrides afterwards; a pytest fixture with `yield` is the natural place.

## Part 6: Structuring an application

A single file with every route works for a demo. A service that several people maintain needs boundaries.

![The API layer and the service layer](../../../assets/diagrams/api-and-service-layers.svg)

**How this works**

- The **API layer** owns everything HTTP: paths, query parameters, status codes, response schemas, exception handlers. Its routes are thin: validate, call the service, shape the response.
- The **service layer** owns the operations (`find`, `get`, `add`, `report`) and raises **domain** exceptions. It imports nothing from FastAPI, so it can be tested with plain pytest and reused from the command line or a job.
- The **domain** code from Sessions 03 to 06 is unchanged. The command line and the API are two front ends on the same logic, which is the payoff of the package structure from Session 04.
- `Depends(get_service)` is the seam between the layers. Tests replace the service through `dependency_overrides`, so API tests can control exactly what the service returns.
- The arrows only point down. If the service layer starts importing `HTTPException`, the boundary has leaked.

### Routers

An `APIRouter` groups related routes, typically one module per resource, with a shared prefix and documentation tag. The application then includes the routers:

In [ ]:
from fastapi import APIRouter

assets = APIRouter(prefix="/assets", tags=["assets"])


@assets.get("/{asset_id}")
def get_asset_routed(asset_id: str, service: ServiceDep) -> AssetOut:
    return service.get(asset_id)


@assets.get("")
def count_assets(service: ServiceDep) -> dict:
    return {"assets": service.count()}


routed_app = FastAPI(title="Asset Inventory")
routed_app.include_router(assets)
routed_app.add_exception_handler(AssetNotFoundError, asset_not_found)

print(sorted(routed_app.openapi()["paths"]))  # the paths the app now serves
print(TestClient(routed_app).get("/assets/db-03").json())

### An app factory and startup work

Two more pieces turn a script into a service:

- An **app factory**, `create_app(settings=None)`, builds and configures the app. Tests call it to get a fresh app with their own settings; production code uses the module-level `app = create_app()` that `fastapi dev` finds.
- A **lifespan** function runs code at **startup** (before the first request) and **shutdown**. It's where the service is created, for example by loading the export files once, so that bad settings stop the server immediately instead of failing on the first request.

The lifespan is written with `async def` and `@asynccontextmanager`. Asynchronous code is Session 08's topic; for now, read it as "everything before `yield` runs at startup, everything after it at shutdown".

In [ ]:
from collections.abc import AsyncIterator
from contextlib import asynccontextmanager


def get_app_service(request: Request) -> AssetService:
    return request.app.state.service  # created at startup, below


AppServiceDep = Annotated[AssetService, Depends(get_app_service)]
api = APIRouter(prefix="/assets", tags=["assets"])


@api.get("/{asset_id}")
def read_asset(asset_id: str, service: AppServiceDep) -> AssetOut:
    return service.get(asset_id)


def create_app(records: dict[str, AssetRecord] | None = None) -> FastAPI:
    @asynccontextmanager
    async def lifespan(app: FastAPI) -> AsyncIterator[None]:
        print("startup: loading assets")
        app.state.service = AssetService(RECORDS if records is None else records)
        yield
        print("shutdown")

    app = FastAPI(title="Asset Inventory", lifespan=lifespan)
    app.include_router(api)
    app.add_exception_handler(AssetNotFoundError, asset_not_found)
    return app


with TestClient(create_app()) as client:  # `with` runs startup and shutdown
    print(client.get("/assets/db-03").json())
    print(client.get("/assets/nope").status_code)

**Predict:** what happens if you use `TestClient(create_app())` **without** `with`, and call `/assets/db-03`?

<details>
<summary>Show answer</summary>

The lifespan never runs, so `app.state.service` doesn't exist and `get_app_service` raises `AttributeError`: a `500`, re-raised by `TestClient`. Only the `with` form runs startup and shutdown. The workbook's `client` fixture uses `with` for this reason.

</details>

`app.state` is a simple namespace for objects that live as long as the app. The provided code in the workbook stores the `TicketService` there.

## Part 7: Testing with pytest

`TestClient` turns API tests into ordinary pytest functions. The usual shape is a fixture that builds the app, overrides what it needs, and yields a client. Here's a small project written to a temporary folder and tested with real pytest:

In [ ]:
import os
import pathlib
import subprocess
import sys
import tempfile
import textwrap

APP = """
from typing import Annotated

from fastapi import Depends, FastAPI, HTTPException, status


def get_inventory() -> dict[str, str]:
    raise RuntimeError("the real inventory isn't available in tests")


Inventory = Annotated[dict[str, str], Depends(get_inventory)]
app = FastAPI()


@app.get("/assets/{asset_id}")
def owner(asset_id: str, inventory: Inventory) -> dict[str, str]:
    if asset_id not in inventory:
        raise HTTPException(status.HTTP_404_NOT_FOUND, f"asset {asset_id} not found")
    return {"id": asset_id, "team": inventory[asset_id]}
"""

TESTS = """
import pytest
from fastapi.testclient import TestClient

from inventory_api import app, get_inventory


@pytest.fixture
def client():
    app.dependency_overrides[get_inventory] = lambda: {"db-03": "database"}
    with TestClient(app) as test_client:
        yield test_client
    app.dependency_overrides.clear()


def test_known_asset(client):
    response = client.get("/assets/db-03")
    assert response.status_code == 200
    assert response.json() == {"id": "db-03", "team": "database"}


def test_unknown_asset_is_404(client):
    response = client.get("/assets/printer-7")
    assert response.status_code == 404
    assert response.json() == {"detail": "asset printer-7 not found"}


@pytest.mark.parametrize("url", ["/assets", "/asset/db-03"])
def test_wrong_paths_are_404(client, url):
    assert client.get(url).status_code == 404
"""

env = {k: v for k, v in os.environ.items() if k not in ("FORCE_COLOR", "CLICOLOR_FORCE")}
env.update(NO_COLOR="1", PY_COLORS="0")
with tempfile.TemporaryDirectory() as workdir:
    folder = pathlib.Path(workdir)
    (folder / "inventory_api.py").write_text(textwrap.dedent(APP), encoding="utf-8")
    (folder / "test_inventory_api.py").write_text(textwrap.dedent(TESTS), encoding="utf-8")
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"],
        cwd=folder,
        capture_output=True,
        text=True,
        env=env,
    )
print(result.stdout)

**Explain:** the real `get_inventory` raises, and the tests still pass, because the fixture overrides it. The tests exercise routing, the path parameter, the `404` and the JSON body, all in memory.

What to test in an API, roughly in order of value:

1. Each endpoint's **success** case: status code **and** body.
2. Each **error status** you promise: `404`, `409`, `401`.
3. **Validation**: one or two `422` cases per input, checking `loc`, so you know the right field was blamed.
4. The **contract**: the OpenAPI document contains the paths, parameters and responses you documented (next part).

Put business rules in the **service** and test them there with plain pytest, which is faster and simpler. API tests then only need to show that HTTP is wired correctly.

## Part 8: Documentation for free

FastAPI generates an **OpenAPI** document (the standard formerly called Swagger) from your routes, parameters and models. A running server serves it at `/openapi.json`, with interactive documentation at **`/docs`** (Swagger UI) and `/redoc`.

In [ ]:
import json

doc_app = FastAPI(title="Asset Inventory", version="0.3.0")
doc_app.include_router(assets)


@doc_app.post(
    "/assets",
    status_code=201,
    tags=["assets"],
    responses={409: {"description": "An asset with this id already exists"}},
)
def add_asset(asset: NewAsset) -> NewAsset:
    return asset


schema = doc_app.openapi()
print(schema["info"])
for path, operations in schema["paths"].items():
    for method, operation in operations.items():
        print(f"{method.upper():<5} {path:<22} responses {sorted(operation['responses'])}")
print(json.dumps(schema["components"]["schemas"]["NewAsset"], indent=2)[:600])

**Explain:**

- Every route, parameter, constraint and model appears in the document. The `NewAsset` schema is `NewAsset.model_json_schema()` from Session 06.
- `422` is documented automatically for every route that validates input. Other error responses are documented only if you declare them with `responses={...}`. FastAPI can't know that a handler turns an exception into `409`.
- The document is generated from the code, so it can't drift out of date the way a hand-written wiki page does. Other teams can generate clients from it.

**Engineering Insight:** `/docs` is an API anyone on the network can read. That's ideal internally; for a public service, decide deliberately whether to expose it (`FastAPI(docs_url=None)` turns it off).

### Running a real server

In a project, you start the server from a terminal, not from a notebook:

```bash
uv run fastapi dev          # development: auto-reload on code changes, http://127.0.0.1:8000
uv run fastapi run          # production mode: no reload, listens on all interfaces
```

`fastapi dev` finds the app through `[tool.fastapi] entrypoint = "package.module:app"` in `pyproject.toml`, which the workbook sets up. Then open `http://127.0.0.1:8000/docs` and try the endpoints in the browser.

## Part 9: `def` or `async def`?

FastAPI accepts both. In this session every endpoint is a plain `def`, which FastAPI runs in a **thread pool**, so one slow request doesn't block the others. `async def` endpoints run on the event loop instead and must never call blocking code. That trade-off is the subject of Session 08.

One consequence matters now: with a thread pool, **two requests can run at the same time**. The workbook keeps tickets in an in-memory dict, which is fine for learning and for one process, but it means:

- data added through the API is **lost on restart**;
- with several worker processes (`fastapi run --workers 4`), each worker has its **own** copy;
- a real service would use a database, which handles concurrency for you.

Name these limitations in your workbook notes; recognising them is part of the job.

## Common mistakes

| Mistake | What happens | Instead |
|---|---|---|
| Returning the internal model from an endpoint | Internal fields leak into responses | A separate response model; the return type filters |
| `KeyError` / `None` for a missing resource | `500` (or `200` with `null`) instead of `404` | A domain exception plus a handler, or `HTTPException(404)` |
| `HTTPException` raised in the service layer | The service can't be reused outside HTTP | Domain exceptions; map them in the API layer |
| A handler for `Exception` that hides failures | Bugs look like successes | Handle expected exceptions; let the rest be `500`s, and log them |
| `POST` that creates returning `200` | Clients can't tell creation from other outcomes | `status_code=201` |
| Building services inside endpoints | Every request repeats the work; tests can't substitute | `Depends(get_service)`, created once at startup |
| Forgetting `with TestClient(app)` | Startup never runs; `app.state` is empty | Use the `with` form whenever the app has a lifespan |
| Leaving `dependency_overrides` set | One test changes the next test's app | Clear them in the fixture's teardown |
| Ignoring unknown query parameters | `?teem=x` silently returns unfiltered data | A filter model with `extra="forbid"` |
| Undocumented error responses | Clients learn about `409` in production | `responses={409: {...}}` on the route |

## Practical engineering application

Every AI service you build after this course has this shape: a FastAPI app whose routes validate input with Pydantic, call a service (which calls an LLM, a vector store or another API), and return a response model. The RAG capstone in Sessions 14 and 15 is exactly that. The habits from today carry over unchanged: **thin routes, a service layer without HTTP, domain exceptions mapped to status codes, dependencies you can override, and tests that run without a server.**

## Key takeaways

- An endpoint is a typed function. FastAPI reads path and query parameters and the body from its parameters, validates them with Pydantic, and answers `422` before your code runs if anything is wrong.
- The return type is the response model: it validates, filters and documents what leaves the service.
- Choose status codes deliberately: `201` for creation, `404`, `409`, and `500` only for bugs. Map domain exceptions with handlers, and keep `HTTPException` in the API layer.
- `Depends` injects settings and services per request; `dependency_overrides` replaces them in tests.
- Structure: routers for HTTP, a service layer without HTTP, and an app factory with a lifespan for startup work.
- `TestClient` tests the API in memory; `with TestClient(app)` runs startup and shutdown. `/docs` and `/openapi.json` come free.

## Further reading

The curated references for this session are in [`../resources/references.md`](../resources/references.md).

**Next:** test yourself with **`02-practice-quiz.ipynb`**, then continue **Project 02** with the workbook.